# Processing Data of Different Modalities
## Real-world data and numerical representations

**Laboratory 2 — individual assignment — maximum 10 points**

Practical 06 introduced image and text representations; Practical 07 covered time series and graphs.
In this laboratory you will work independently with **two real datasets**. The challenge is to
choose numerical representations that help answer a question without hiding important structure.

### Choose your datasets

| Group A — choose one | Group B — choose one |
|---|---|
| **Handwritten digits** — 1,797 scanned digit images | **Mauna Loa CO₂** — historical weekly measurements |
| **Movie review snippets** — 200 labelled sentences | **Karate Club** — 34 people and observed social ties |

Complete **one** dataset from each column, not all four. Each option has its own task and source.
The datasets are provided locally so **no web download is needed** when running this notebook.

You do not need PCA, clustering, machine learning classifiers or neural networks. Those come later.

### Deliverable

Submit one completed notebook named `Lab 02 - <Surname> <Name>.ipynb`.

For **each** chosen dataset, include an inspection, appropriate numerical representations,
a numerical comparison, a figure and a short interpretation. Discuss at least one weakness of
what you built using evidence from the data.

Use `Restart Kernel → Run All` before submission. Keep original files unchanged, use relative
paths and do not put known outcome labels into the numerical features being evaluated.

**Read [`SOURCES_AND_DATA_DICTIONARY.md`](SOURCES_AND_DATA_DICTIONARY.md)** before starting.
The four sources have different units, collection methods and limitations.

---
## 0. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path("data")

# Choose exactly one from each group.
GROUP_A = "images"       # "images" or "text"
GROUP_B = "time_series"  # "time_series" or "graphs"

assert GROUP_A in {"images", "text"}
assert GROUP_B in {"time_series", "graphs"}

files = {
    "images": ["handwritten_digits.npz"],
    "text": ["movie_review_snippets.csv"],
    "time_series": ["mauna_loa_co2_weekly.csv"],
    "graphs": ["karate_edges.csv", "karate_members.csv"],
}

missing = [name for name in files[GROUP_A] + files[GROUP_B]
           if not (DATA / name).is_file()]
if missing:
    raise FileNotFoundError(f"Missing files in data/: {missing}")

print("Chosen datasets:", GROUP_A, "+", GROUP_B)

### Your questions

Before coding, write one question that you can answer with each dataset.
Be specific about what you want to compare.

**Group A question:** *Write here.*

**Group B question:** *Write here.*

---
## Group A — choose Images or Text

### Option A1 — Handwritten digits: does an intensity histogram describe the shape?

The digits are **real handwritten samples**, not generated geometric patterns. The UCI dataset was
constructed by reducing original 32×32 digit bitmaps to an 8×8 grid of counts (0–16); this package
uses the 1,797-image test subset distributed with `scikit-learn`.

Explore the images first. Display at least six digits with their labels and inspect their shapes,
dtypes and intensity values. The digit labels are there **only to check what your comparisons find**.

Then compare two ways to represent each image:

- an **intensity histogram**, which ignores where the strokes are;
- a **spatial representation** of your choice: flattened pixels, row/column projections, quadrants,
  or another justified feature vector.

Choose a distance measure and compare several image pairs (including at least one pair with
different digit labels). Find evidence that one representation is more useful for distinguishing
shapes, or explain where both struggle. Make a figure that supports your conclusion.

**Discuss:** Does your chosen spatial representation tolerate shifts or different handwriting styles?

In [ ]:
if GROUP_A == "images":
    with np.load(DATA / "handwritten_digits.npz", allow_pickle=False) as archive:
        images = archive["images"]
        digit_labels = archive["labels"]
    print("Images:", images.shape, "Labels:", digit_labels.shape)

In [ ]:
# Your solution starts here.  # Images

**Images — findings:** *Write your interpretation and one limitation here.*

### Option A2 — Movie reviews: when does lexical similarity agree with sentiment?

These are **real review snippets** from the Pang–Lee movie-review sentiment data, included as a
small, balanced teaching excerpt (100 positive and 100 negative). Unlike the fictional dispatch
reports from the previous version, they contain ordinary vocabulary, varied writing styles and
ambiguous opinions.

Inspect the texts, label balance, duplicate messages and typical document lengths. Then build:

- a **unigram count or TF–IDF representation**;
- a version that **includes word bigrams**.

For several documents, inspect nearest neighbours under cosine similarity **without comparing
an item to itself**. Find at least one case where lexical similarity and the provided sentiment
labels disagree. Does including bigrams change your finding?

Make one figure comparing the selected documents (for example, a small cosine-similarity heatmap).
Explain why the labels are useful for evaluation but must not be used as document-vector features.

For this exploratory task, fitting the vectorizers to the whole small corpus is acceptable.
For predictive modelling, the vocabulary and IDF statistics must be fitted on training data only.

In [ ]:
if GROUP_A == "text":
    from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity

    reviews = pd.read_csv(DATA / "movie_review_snippets.csv")
    print("Review snippets:", reviews.shape)

In [ ]:
# Your solution starts here.  # Text

**Text — findings:** *Write your observations and one limitation here.*

---
## Group B — choose Time Series or Graphs

### Option B1 — Mauna Loa: what can a monthly summary tell us about CO₂?

This is a historical snapshot of **measured atmospheric CO₂ concentrations** from Mauna Loa,
Hawaii, in parts per million (ppm), covering 1958–2001. It is **not current data**.
The weekly series contains genuine missing measurements.

Parse the dates, verify their ordering and count missing values. Plot the full series before
performing any smoothing. Then:

1. Compute **monthly means** and a companion count of available weekly observations per month.
   Explain how you handle months with missing observations.
2. Create a **time-aware feature** that makes one useful pattern clearer, such as a 12-month
   rolling mean, annual summaries or year-over-year differences.
3. Compare **two time periods** of your choice (for example, a period in the 1960s and one in the
   1990s) using at least one numerical result and one figure.

Conclude what monthly aggregation or smoothing reveals and what it hides. Do not interpret missing
weeks as measurements of zero. Do not claim a causal explanation based on this one series alone.

In [ ]:
if GROUP_B == "time_series":
    co2 = pd.read_csv(DATA / "mauna_loa_co2_weekly.csv")
    print("CO₂ observations:", co2.shape)

In [ ]:
# Your solution starts here.  # Time series

**Time series — findings:** *Write your result, missing-data choice and one limitation here.*

### Option B2 — Karate Club: what can local node features tell us about a network?

The **Zachary Karate Club** network records observed social ties between 34 club members in the
1970s. Edges are undirected. The edge weight is an **interaction-context count**, *not* distance,
travel time or an automatic shortest-path cost. Member labels describe which club each person
joined **after the split**, so they are outcomes for interpretation, not inputs when building the graph.

Build the full graph from `karate_edges.csv` and `karate_members.csv`. Report nodes, edges,
components and the meaning of the edge weights. Then:

- create an **unweighted adjacency matrix** with an explicit node order;
- build a **node-feature table** with degree and one additional structural descriptor, such as
  local clustering or betweenness centrality;
- identify the most connected members and compare whether your second descriptor tells the same story;
- find a **bridge** (if one exists), remove it in a *copy* of the graph, and show what happens to
  connected components.

Include a network plot or an adjacency-matrix image. Explain why a table of node degrees alone
cannot reconstruct the original social network.

You may use `club_after_split` to interpret the result, but not to invent edges or as a node feature
that is supposed to describe the original graph.

In [ ]:
if GROUP_B == "graphs":
    import networkx as nx

    karate_edges = pd.read_csv(DATA / "karate_edges.csv")
    karate_members = pd.read_csv(DATA / "karate_members.csv")
    print("Edges:", karate_edges.shape, "Members:", karate_members.shape)

In [ ]:
# Your solution starts here.  # Graphs

**Graphs — findings:** *Write your result and one structural limitation here.*

---
## 3. Compare your two approaches

Only compare the **two datasets you selected**. You are not expected to compare image pixels directly
to graph nodes or CO₂ measurements to words.

| | Group A | Group B |
|---|---|---|
| What is one original observation? | *Your answer* | *Your answer* |
| Which structure matters? | *Your answer* | *Your answer* |
| Which numerical representation did you use? | *Your answer* | *Your answer* |
| What did you actually compare? | *Your answer* | *Your answer* |
| What important detail was lost or hidden? | *Your answer* | *Your answer* |

Finish with a **short conclusion (100–150 words)** referring to actual outputs from your notebook.
What did the representations help you discover, and when might they give a misleading answer?

**Conclusion:** *Write here.*

---
## 4. Grading — 10 points

| Criterion | Points | Evidence |
|---|---:|---|
| **Questions and initial inspection** | **1.0** | Useful questions, correct source and data types |
| **Group A** | **3.0** | Numerical representations, comparison and interpretation |
| **Group B** | **3.0** | Respecting temporal / graph structure, comparison and interpretation |
| **Figures** | **1.0** | One readable and relevant figure for each selected modality |
| **Final discussion** | **1.0** | Specific preserved and lost properties, grounded in outputs |
| **Reproducibility** | **1.0** | Fresh-kernel run, unchanged source data, correct separation of labels/features |
| **Total** | **10.0** | |

### Before submitting

- [ ] One Group A and one Group B option completed.
- [ ] Both analyses contain calculated numerical comparisons and at least one figure.
- [ ] The source and collection limitations are acknowledged.
- [ ] No manual or invented numerical results.
- [ ] Original files unchanged; **Restart Kernel → Run All** succeeds.

### Source references

All citations and redistribution notes are in `SOURCES_AND_DATA_DICTIONARY.md`.

- [UCI — Optical Recognition of Handwritten Digits](https://archive.ics.uci.edu/dataset/80/optical+recognition+of+handwritten+digits)
- [Cornell — Movie Review Data](https://www.cs.cornell.edu/people/pabo/movie-review-data/)
- [statsmodels — Mauna Loa weekly atmospheric CO₂](https://www.statsmodels.org/stable/datasets/generated/co2.html)
- [NetworkX — Zachary Karate Club](https://networkx.org/documentation/stable/reference/generated/networkx.generators.social.karate_club_graph.html)